In [ ]:
# 1. GPU check. Runtime -> Change runtime type -> GPU. Use the same GPU type for every run.
!nvidia-smi

In [ ]:
# 2. Google Drive: data, results and logs/ are kept in MyDrive/llmrouter_encoder/ so a disconnect loses nothing.
#    After a disconnect, run cells 1-6 again, then the cell you were on.
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['WORK'] = '/content/drive/MyDrive/llmrouter_encoder'   # survives the session
os.environ['REPO'] = '/content/llm-routing-study'                 # fast local disk, wiped with the session
os.makedirs(os.environ['WORK'], exist_ok=True)

In [ ]:
%%bash
# 3. Code: the encoder branch.
set -e
if [ -d "$REPO/.git" ]; then
  cd "$REPO" && git fetch -q && git checkout -q encoder && git pull -q
else
  git clone -q -b encoder https://github.com/rahulkanagaraj786/llm-routing-study.git "$REPO"
fi
cd "$REPO" && git log --oneline -1

In [ ]:
%%bash
# 4. Packages. Removes Colab's torchaudio/torchvision/torchcodec (they no longer match the installed torch).
cd "$REPO"
pip install -q -r requirements.txt
pip uninstall -y -q torchaudio torchvision torchcodec
if ! python -c "import torch, sys; sys.exit(0 if torch.cuda.is_available() else 1)" 2>/dev/null; then
  echo "The pinned torch build cannot use this GPU driver; installing torch 2.14.1 for CUDA 12.6"
  pip install -q torch==2.14.1 --index-url https://download.pytorch.org/whl/cu126
fi
python -c "import torch, transformers; from transformers.models.modernbert import modeling_modernbert; print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', torch.cuda.get_device_name(0), '| native bf16:', torch.cuda.is_bf16_supported(including_emulation=False))"

In [ ]:
%%bash
set -e
cd "$REPO"
mkdir -p data/processed
if [ -f "$WORK/data/processed/paired_v2.jsonl.gz" ]; then
  cp "$WORK"/data/processed/* data/processed/
  echo "data copied from Drive"
else
  mkdir -p data/raw/extracted
  curl -sSL -o data/raw/bench-release.tar.gz "https://huggingface.co/datasets/NPULH/LLMRouterBench/resolve/0e5af1b84bf73437a01a1849c0f1d2468baa93fc/bench-release.tar.gz"
  echo "b79f8cde1a6f029c2efa663a3a3b6f7748defb22341fe59f328cebef6648c8f1  data/raw/bench-release.tar.gz" | sha256sum -c
  tar -xzf data/raw/bench-release.tar.gz -C data/raw/extracted -T data/selected_members.txt
  python scripts/01_build_paired.py | tail -3
  mkdir -p "$WORK/data/processed" && cp data/processed/* "$WORK/data/processed/"
  rm -rf data/raw
  echo "data built and saved to Drive"
fi

In [ ]:
%%bash
# 6. Keep results folders and logs/ on Drive.
set -e
cd "$REPO"
for d in results/dev_selection/encoder results/final_dryrun/encoder results/final/encoder logs; do
  mkdir -p "$WORK/$d" "$(dirname "$d")"
  if [ -d "$d" ] && [ ! -L "$d" ]; then cp -rn "$d"/. "$WORK/$d"/ && rm -rf "$d"; fi   # keep anything already written
  [ -L "$d" ] || ln -s "$WORK/$d" "$d"
  echo "$d -> $(readlink "$d")"
done

In [ ]:
%%bash
# 7. Tests and smoke check: epoch lines should end with 'cuda' and it should print SMOKE OK.
cd "$REPO"
python tests/test_policy.py && python tests/test_guards.py && python tests/test_models.py
python scripts/05_dev_compare.py encoder --smoke

In [ ]:
%%bash
# 8. Development run: six fine-tunes, resumes from the last finished rotation.
#    Then copy MyDrive/llmrouter_encoder/results/dev_selection/encoder/ into the repo and commit it before cell 10.
cd "$REPO"
export TRANSFORMERS_VERBOSITY=error HF_HUB_DISABLE_PROGRESS_BARS=1 TOKENIZERS_PARALLELISM=false
n_done=$(ls results/dev_selection/encoder/rotation_*.json 2>/dev/null | wc -l | tr -d ' ')
echo "Started $(date '+%H:%M:%S')"
echo "Plan: 5 rotations = 6 fine-tunes (rotation 0 tries lr 2e-05 and 5e-05, rotations 1-4 reuse the better one)."
echo "Rotations already finished and saved on Drive: $n_done of 5"
# fine-tunes left in this session (a half-done rotation restarts from its beginning)
if [ "$n_done" -eq 0 ]; then export ENCODER_TOTAL_FITS=6; else export ENCODER_TOTAL_FITS=$((5 - n_done)); fi
export ENCODER_PROGRESS_SECONDS=20   # a progress line every 20 seconds
echo "Fine-tunes to run now: $ENCODER_TOTAL_FITS"
python -u scripts/05_dev_compare.py encoder 2>&1 \
  | grep --line-buffered -v "unauthenticated requests to the HF Hub" \
  | tee -a "$WORK/dev_encoder.log"
echo "Finished $(date '+%H:%M:%S'). Rotations saved: $(ls results/dev_selection/encoder/rotation_*.json 2>/dev/null | wc -l | tr -d ' ') of 5"


In [ ]:
%%bash
# 9. Rehearsal of the final evaluation on development questions only.
cd "$REPO"
python scripts/07_final_eval.py encoder --plan
python scripts/07_final_eval.py encoder --dry-run 2>&1 | tee -a "$WORK/dryrun_encoder.log"
python scripts/09_report_final.py encoder --dry-run

In [ ]:
TEAM_AGREED = False   # set to True only after the team agrees to run the final evaluation

import subprocess
if not TEAM_AGREED:
    print("Not run. Set TEAM_AGREED = True once the team has agreed.")
else:
    subprocess.run('cd "$REPO" && python scripts/07_final_eval.py encoder --unlock-final 2>&1 | tee -a "$WORK/final_encoder.log" '
                   '&& python scripts/09_report_final.py encoder', shell=True, check=True, executable='/bin/bash')